In [ ]:
"""
Run 1 - Prototype-based Explainable Statute Prediction with InLegalBERT
PROTOTYPE-CONTRASTIVE FINE-TUNING (InfoNCE)
"""

import subprocess
import sys
import importlib


def ensure_packages():
    pkgs = {
        "torch": "torch", "transformers": "transformers", "scikit-learn": "sklearn",
        "nltk": "nltk", "numpy": "numpy", "pandas": "pandas",
        "scikit-multilearn": "skmultilearn", "tqdm": "tqdm", "pysbd": "pysbd",
        "rank_bm25": "rank_bm25",
    }
    for pip_name, import_name in pkgs.items():
        try:
            importlib.import_module(import_name)
        except ImportError:
            print(f"[setup] installing {pip_name} ...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pip_name], check=True)


ensure_packages()

import os
import re
import json
import math
import random
import difflib
import copy
import warnings
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

import pysbd
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    hamming_loss, classification_report,
)
from transformers import AutoTokenizer, AutoModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE, "| AMP:", USE_AMP)


# --------------------------------------------------------------------------
#  Configuration
# --------------------------------------------------------------------------
@dataclass
class ProtoConfig:
    # data
    train_path: str = "task1.jsonl"                        # {doc_id, fact, statute, explanation}
    prototype_source_path: str = "ipc_sections_clean.json"  # full IPC section catalog

    # shared encoder
    encoder_name: str = "law-ai/InLegalBERT"
    freeze_layers: int = 0            
    max_length: int = 384             
    max_sentences: int = 60

    # prototype-contrastive training (Run 1)
    temperature: float = 0.05
    num_hard_negatives: int = 8        # mined hard-negative prototypes per positive
    num_random_negatives: int = 8      # + random global negatives, refreshed every epoch
    exclude_same_base_section: bool = True   
    encoder_lr: float = 2e-5
    weight_decay: float = 0.01
    batch_size: int = 8
    grad_accum_steps: int = 2
    num_epochs: int = 15               
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    use_class_balanced_sampling: bool = True

    # split
    train_fraction: float = 0.70
    val_fraction: float = 0.10
    test_fraction: float = 0.20

   
    use_whitening: bool = True
    whitening_dim: int = 256

    
    evidence_per_prototype: int = 2     # used only by the legacy max-sim evidence in score_case()
    cutoff_grid: tuple = tuple(round(x, 2) for x in np.arange(-0.20, 0.81, 0.02))
    margin_grid: tuple = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15)
    top_k_fallback: int = 1

    # --- evidence sentence retrieval: BM25 + Cosine + Classifier-Based Relevance ---
    top_m_evidence: int = 3             # Top-m evidence sentences per predicted IPC
    evidence_bm25_weight: float = 0.30
    evidence_cosine_weight: float = 0.40
    evidence_classifier_weight: float = 0.30
    relevance_clf_hidden: int = 128
    relevance_clf_epochs: int = 5
    relevance_clf_lr: float = 1e-3
    relevance_clf_batch_size: int = 64
    relevance_clf_neg_per_pos: int = 4   # sampled negative statutes per positive (sentence, statute) pair

    # explanation generator (LLM-Based Reasoning Generation, with CoT prompting)
    use_qwen_explainer: bool = False
    qwen_model_name: str = "Qwen/Qwen2.5-1.5B-Instruct"
    qwen_max_new_tokens: int = 220

    out_dir: str = "./prototype_contrastive_outputs"


cfg = ProtoConfig()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)


# --------------------------------------------------------------------------
#  Case data and Statute Prototype Bank
# --------------------------------------------------------------------------
def load_jsonl_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stat = rec.get("statute", [])
            if isinstance(stat, str):
                stat = [stat]
            rec["statute"] = [str(s).strip() for s in stat]
            records.append(rec)
    return records


def normalize_ipc_label(label):
    m = re.search(r"(\d+[A-Za-z]*)", str(label).strip())
    return f"IPC {m.group(1).upper()}" if m else None


def base_number_of(section_code):
    m = re.search(r"(\d+)", section_code)
    return m.group(1) if m else section_code


def load_statute_prototypes(path):
    """Returns (prototype_texts {code: description}, prototype_titles {code: title})."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    texts, titles = {}, {}

    def store(code_raw, text, title=None):
        code = normalize_ipc_label(code_raw)
        if code and text and str(text).strip():
            texts[code] = str(text).strip()
            if title:
                titles[code] = str(title).strip()

    def pick_text(d):
        return (d.get("description") or d.get("text") or d.get("content") or d.get("definition")
                or d.get("summary") or d.get("section_desc") or "")

    def pick_title(d):
        return d.get("title") or d.get("name") or d.get("offense") or d.get("heading")

    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                store(k, v)
            elif isinstance(v, dict):
                store(k, pick_text(v), pick_title(v))
    elif isinstance(raw, list):
        for item in raw:
            if not isinstance(item, dict):
                continue
            code_raw = (item.get("section") or item.get("section_number") or item.get("code")
                        or item.get("ipc_section") or item.get("id") or item.get("section_no"))
            if code_raw is not None:
                store(code_raw, pick_text(item), pick_title(item))
    else:
        raise ValueError(f"Unrecognised prototype file schema: {type(raw)}")
    return texts, titles


records = load_jsonl_dataset(cfg.train_path)
prototype_texts, prototype_titles = load_statute_prototypes(cfg.prototype_source_path)
prototype_codes = sorted(prototype_texts.keys())
print(f"{len(records)} cases | {len(prototype_codes)} statute prototypes in full catalog")
for c in prototype_codes[:3]:
    print(f"  {c}: {prototype_texts[c][:110]}...")
if not prototype_codes:
    print("WARNING: 0 prototypes parsed - check the field names in load_statute_prototypes().")

gold_seen = sorted({normalize_ipc_label(s) for r in records for s in r["statute"]} - {None})
missing = sorted(set(gold_seen) - set(prototype_codes))
if missing:
    print("WARNING: gold sections absent from the Prototype Bank:", missing)
print(f"{len(gold_seen)} supervised sections out of {len(prototype_codes)} prototypes")


active_codes = [c for c in prototype_codes if c in set(gold_seen)]
if not active_codes:
    print("WARNING: no overlap between gold labels and prototype catalog; "
          "falling back to the full catalog as the active label set.")
    active_codes = list(prototype_codes)
print(f"Restricting candidate labels to {len(active_codes)} sections actually "
      f"observed in gold data (of {len(prototype_codes)} total prototypes).")



_segmenter = pysbd.Segmenter(language="en", clean=False)


def split_sentences(text, max_sentences=None):
    sents = [s.strip() for s in _segmenter.segment(text or "") if s.strip()]
    if not sents:
        sents = [text.strip()] if text and text.strip() else ["."]
    return sents[:max_sentences] if max_sentences else sents


def positive_pairs_from_explanation(fact, explanation, gold_sections):
    """(case sentence, positive prototype code) pairs, built from the per-sentence
    `explanation` field (exact match first, fuzzy fallback). If the explanation
    dict yields nothing for a case that DOES have gold labels, fall back to
    pairing every sentence with every gold label, so no case is wasted.
    Reused both for InfoNCE training pairs and for training the evidence
    relevance classifier."""
    sentences = split_sentences(fact, cfg.max_sentences)
    pairs = []
    for exp_sent, label in (explanation or {}).items():
        code = normalize_ipc_label(label)
        if not code or code not in prototype_texts:
            continue
        exp_norm = re.sub(r"\s+", " ", exp_sent).strip()
        best_s, best_r = None, 0.0
        for s in sentences:
            r = difflib.SequenceMatcher(None, re.sub(r"\s+", " ", s).strip(), exp_norm, autojunk=False).ratio()
            if r > best_r:
                best_r, best_s = r, s
        if best_s is not None and best_r >= 0.5:
            pairs.append((best_s, code))
    if not pairs and gold_sections:
        for code in gold_sections:
            if code in prototype_texts:
                for s in sentences:
                    pairs.append((s, code))
    return pairs


for rec in tqdm(records, desc="PySBD"):
    rec["sentences"] = split_sentences(rec["fact"], cfg.max_sentences)
    rec["gold_sections"] = sorted({s for s in (normalize_ipc_label(g) for g in rec["statute"]) if s})

label_counts = Counter(s for r in records for s in r["gold_sections"])
print(dict(sorted(label_counts.items(), key=lambda x: -x[1])))


# --------------------------------------------------------------------------
#  Train / validation / test split (70 / 10 / 20, multilabel-stratified)
# --------------------------------------------------------------------------
def multilabel_stratified_split(docs, fraction, seed):
    from skmultilearn.model_selection import iterative_train_test_split
    labels = sorted({s for d in docs for s in d["gold_sections"]})
    y = MultiLabelBinarizer(classes=labels).fit_transform([d["gold_sections"] for d in docs])
    X = np.arange(len(docs)).reshape(-1, 1)
    np.random.seed(seed)
    X_keep, _, X_held, _ = iterative_train_test_split(X, y, test_size=fraction)
    keep, held = set(X_keep.flatten().tolist()), set(X_held.flatten().tolist())
    return [docs[i] for i in range(len(docs)) if i in keep], [docs[i] for i in range(len(docs)) if i in held]


remainder, test_split = multilabel_stratified_split(records, cfg.test_fraction, SEED)
train_split, val_split = multilabel_stratified_split(
    remainder, cfg.val_fraction / (cfg.train_fraction + cfg.val_fraction), SEED + 1)
print(f"Train {len(train_split)} | Val {len(val_split)} | Test {len(test_split)}")


# ---------------------------------------
#  Shared InLegalBERT prototype encoder 
# ---------------------------------------
def mean_pool(hidden, mask):
    m = mask.unsqueeze(-1).float()
    return (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)


class PrototypeEncoder(nn.Module):
    """Shared InLegalBERT encoder producing L2-normalised embeddings for sentences and prototypes."""

    def __init__(self, model_name, freeze_layers=0):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.embed_dim = self.bert.config.hidden_size
        self.freeze_bottom(freeze_layers)

    def freeze_bottom(self, n):
        if n <= 0:
            return
        for p in self.bert.embeddings.parameters():
            p.requires_grad = False
        for i, layer in enumerate(self.bert.encoder.layer):
            for p in layer.parameters():
                p.requires_grad = i >= n
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Encoder: {trainable:,} trainable parameters (bottom {n} layers frozen)")

    def forward(self, texts, max_length):
        enc = self.tokenizer(texts, truncation=True, padding=True, max_length=max_length,
                              return_tensors="pt").to(next(self.parameters()).device)
        hidden = self.bert(**enc).last_hidden_state
        return F.normalize(mean_pool(hidden, enc["attention_mask"]), p=2, dim=-1)

    @torch.no_grad()
    def embed(self, texts, max_length, batch_size=32):
        self.eval()
        out = [self(texts[i:i + batch_size], max_length).float().cpu()
               for i in range(0, len(texts), batch_size)]
        return torch.cat(out, 0) if out else torch.zeros((0, self.embed_dim))


encoder = PrototypeEncoder(cfg.encoder_name, cfg.freeze_layers).to(DEVICE)


# --------------------------------------------------------------------------
# Statute Prototype Bank (full catalog, for training-time hard negatives)
#    + hard-negative prototype mining
# --------------------------------------------------------------------------
def build_prototype_bank(enc):
    """Embeddings of ALL statute prototypes in the full catalog, shape (num_prototypes, dim).
    Used only for hard-negative mining during training -- deliberately NOT restricted
    to active_codes, since broader/harder negatives are still useful regularization."""
    return enc.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)


def mine_hard_negative_prototypes(bank_emb, k, exclude_same_base=True):
    """For every prototype, pick the k most similar prototypes that are LEGALLY
    DIFFERENT (a different base IPC section) -- these are its hard negatives."""
    sim = bank_emb @ bank_emb.t()
    hard = {}
    for i, code in enumerate(prototype_codes):
        negatives = []
        for j in torch.argsort(sim[i], descending=True).tolist():
            other = prototype_codes[j]
            if j == i:
                continue
            if exclude_same_base and base_number_of(other) == base_number_of(code):
                continue
            negatives.append(other)
            if len(negatives) == k:
                break
        hard[code] = negatives
    return hard


# --------------------------------------------------------------------------
#  Whitening -- optional post-hoc anisotropy cleanup on top of fine-tuning
# --------------------------------------------------------------------------
_whitening = {"mu": None, "W": None}


def fit_whitening(reference_embeddings, target_dim=None):
    X = reference_embeddings.double()
    mu = X.mean(dim=0, keepdim=True)
    Xc = X - mu
    cov = (Xc.t() @ Xc) / (Xc.shape[0] - 1)
    U, S, _ = torch.linalg.svd(cov)
    W = U @ torch.diag(1.0 / torch.sqrt(S + 1e-6))
    if target_dim:
        W = W[:, :target_dim]
    return mu.float(), W.float()


def apply_whitening(embeddings):
    if not cfg.use_whitening or _whitening["mu"] is None:
        return embeddings
    out = (embeddings - _whitening["mu"]) @ _whitening["W"]
    return F.normalize(out, p=2, dim=-1)


def embed_and_whiten(texts, max_length):
    raw = encoder.embed(texts, max_length)
    return apply_whitening(raw)


def fit_whitening_from_current_encoder():
    # Whitening reference still draws on the full catalog + train sentences for a
    # robust covariance estimate; this is unrelated to the active-label prediction fix.
    raw_prototype_emb = encoder.embed([prototype_texts[c] for c in prototype_codes], cfg.max_length)
    sample_sents = [s for rec in train_split for s in rec["sentences"]][:4000]
    raw_sent_emb = encoder.embed(sample_sents, cfg.max_length) if sample_sents else torch.zeros((0, encoder.embed_dim))
    reference = torch.cat([raw_prototype_emb, raw_sent_emb], dim=0)
    _whitening["mu"], _whitening["W"] = fit_whitening(reference, cfg.whitening_dim)


# --------------------------------------------------------------------------
# 8. Scoring and prediction -- restricted to active_codes (THE FIX),
#    global cutoff + margin, 
#    This is the "Sentence-Statute Similarity Matrix" + "Statute Ranking
#    (Max Sentence Similarity)" 
# --------------------------------------------------------------------------
def build_bank():
    """Embeddings of only the ACTIVE prototypes -- the fixed candidate set used
    for every prediction, at both validation-calibration time and test time."""
    return embed_and_whiten([prototype_texts[c] for c in active_codes], cfg.max_length)


def score_case(fact_text, bank_emb):
    sentences = split_sentences(fact_text, cfg.max_sentences)
    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    sim_matrix = (sent_emb @ bank_emb.t()).numpy()
    best = sim_matrix.max(axis=0)
    scores = {c: float(best[j]) for j, c in enumerate(active_codes)}
    evidence = {}
    for j, c in enumerate(active_codes):
        top_idx = np.argsort(-sim_matrix[:, j])[:cfg.evidence_per_prototype]
        evidence[c] = [sentences[i] for i in top_idx]
    return scores, evidence, sentences


def _predict_from_scores(scores, cutoff, margin):
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    top_code, top_score = ranked[0]
    if top_score < cutoff:
        return [ranked[i] for i in range(min(cfg.top_k_fallback, len(ranked)))]
    chosen = [(top_code, top_score)]
    for c, s in ranked[1:]:
        if s >= top_score - margin and s >= cutoff:
            chosen.append((c, s))
    return chosen


def calibrate_decision_rule(bank_emb):
    """Grid-search (global cutoff, top-score margin) to directly MAXIMISE
    validation macro-F1, over the FIXED active_codes label set -- the same
    set used later in evaluate_run(), so val and test macro-F1 are
    apples-to-apples and calibration can't be blind to over-prediction noise."""
    gold = [d["gold_sections"] for d in val_split]
    val_scores = [score_case(d["fact"], bank_emb)[0] for d in val_split]
    mlb = MultiLabelBinarizer(classes=active_codes)
    yt = mlb.fit_transform(gold)

    best = {"macro_f1": -1.0, "cutoff": cfg.cutoff_grid[0], "margin": cfg.margin_grid[0]}
    for cutoff in cfg.cutoff_grid:
        for margin in cfg.margin_grid:
            preds = [[c for c, _ in _predict_from_scores(sc, cutoff, margin)] for sc in val_scores]
            yp = mlb.transform(preds)
            f1 = f1_score(yt, yp, average="macro", zero_division=0)
            if f1 > best["macro_f1"]:
                best = {"macro_f1": f1, "cutoff": float(cutoff), "margin": float(margin)}
    return best["cutoff"], best["margin"], best["macro_f1"]


# --------------------------------------------------------------------------
# 8b. Evidence Sentence Retrieval: BM25 + Cosine Similarity +
#     Classifier-Based Relevance -> Top-m Evidence Sentences.

# --------------------------------------------------------------------------
def _bm25_tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def bm25_sentence_scores(sentences, query_text):
    """BM25 lexical relevance of each of THIS case's sentences against a
    statute's title+description. The BM25 index is built per-case (over
    just that case's own sentences) so the score reflects which of the
    case's sentences best match the statute, not corpus-wide term rarity."""
    if not sentences:
        return np.zeros(0)
    bm25 = BM25Okapi([_bm25_tokenize(s) for s in sentences])
    scores = np.array(bm25.get_scores(_bm25_tokenize(query_text)), dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / span if span > 1e-9 else np.zeros_like(scores)


def _minmax(x):
    x = np.asarray(x, dtype=np.float64)
    span = x.max() - x.min()
    return (x - x.min()) / span if span > 1e-9 else np.zeros_like(x)


class RelevanceClassifier(nn.Module):
    """Classifier-Based Relevance head: P(sentence supports statute) from
    [sent_emb ; proto_emb ; sent_emb * proto_emb] using the same (whitened)
    InLegalBERT embeddings already computed for the similarity matrix."""

    def __init__(self, dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 3, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, 1),
        )

    def forward(self, sent_emb, proto_emb):
        feat = torch.cat([sent_emb, proto_emb, sent_emb * proto_emb], dim=-1)
        return self.net(feat).squeeze(-1)


def train_relevance_classifier(bank_emb):
    """Trains RelevanceClassifier on (sentence, statute) pairs from the same
    `explanation`-derived positives used for InfoNCE training (Section 4),
    with `relevance_clf_neg_per_pos` sampled negative statutes per positive.
    Frozen embeddings from the FINAL fine-tuned+whitened encoder are used as
    features, so this trains fast (a few epochs of a small MLP)."""
    pos_pairs = []
    for rec in train_split:
        pos_pairs.extend(positive_pairs_from_explanation(
            rec["fact"], rec.get("explanation", {}) or {}, rec["gold_sections"]))
    pos_pairs = [(s, c) for s, c in pos_pairs if c in active_codes]
    if not pos_pairs:
        print("[relevance-classifier] no (sentence, statute) positives found -- "
              "evidence retrieval will fall back to BM25 + cosine only.")
        return None

    code_to_idx = {c: i for i, c in enumerate(active_codes)}
    uniq_sentences = list({s for s, _ in pos_pairs})
    sent_emb_lookup = {s: e for s, e in zip(uniq_sentences, embed_and_whiten(uniq_sentences, cfg.max_length))}

    X_sent, X_proto, y = [], [], []
    for s, c in pos_pairs:
        X_sent.append(sent_emb_lookup[s]); X_proto.append(bank_emb[code_to_idx[c]]); y.append(1.0)
        neg_pool = [cc for cc in active_codes if cc != c]
        for nc in random.sample(neg_pool, min(cfg.relevance_clf_neg_per_pos, len(neg_pool))):
            X_sent.append(sent_emb_lookup[s]); X_proto.append(bank_emb[code_to_idx[nc]]); y.append(0.0)

    X_sent = torch.stack(X_sent).to(DEVICE)
    X_proto = torch.stack(X_proto).to(DEVICE)
    y = torch.tensor(y, dtype=torch.float32, device=DEVICE)

    clf = RelevanceClassifier(X_sent.shape[-1], cfg.relevance_clf_hidden).to(DEVICE)
    opt = torch.optim.Adam(clf.parameters(), lr=cfg.relevance_clf_lr)

    clf.train()
    for epoch in range(1, cfg.relevance_clf_epochs + 1):
        idx = torch.randperm(len(y), device=DEVICE)
        total, steps = 0.0, 0
        for i in range(0, len(y), cfg.relevance_clf_batch_size):
            b = idx[i:i + cfg.relevance_clf_batch_size]
            opt.zero_grad()
            logits = clf(X_sent[b], X_proto[b])
            loss = F.binary_cross_entropy_with_logits(logits, y[b])
            loss.backward()
            opt.step()
            total += loss.item(); steps += 1
        print(f"  [relevance-classifier] epoch {epoch}/{cfg.relevance_clf_epochs} "
              f"BCE loss = {total / max(1, steps):.4f}")
    clf.eval()
    return clf


@torch.no_grad()
def classifier_relevance_scores(clf, sentences, proto_vec):
    if clf is None or not sentences:
        return np.zeros(len(sentences))
    sent_emb = embed_and_whiten(sentences, cfg.max_length).to(DEVICE)
    proto_batch = proto_vec.to(DEVICE).unsqueeze(0).expand(len(sentences), -1)
    probs = torch.sigmoid(clf(sent_emb, proto_batch)).cpu().numpy()
    return probs


def retrieve_evidence_sentences(sentences, section_code, bank_emb, relevance_clf, top_m=None):
    """Score S = {S1..Sn} via BM25 + Cosine Similarity + Classifier-Based
    Relevance, min-max normalise each signal, combine with the configured
    weights, and return the Top-m evidence sentences (sentence, combined_score)
    for `section_code`, ranked highest first."""
    top_m = top_m or cfg.top_m_evidence
    if not sentences:
        return []

    proto_idx = active_codes.index(section_code)
    proto_vec = bank_emb[proto_idx]

    sent_emb = embed_and_whiten(sentences, cfg.max_length)
    cosine_raw = (sent_emb @ proto_vec.unsqueeze(-1)).squeeze(-1).numpy()
    cosine = _minmax(cosine_raw)

    query_text = f"{prototype_titles.get(section_code, '')} {prototype_texts.get(section_code, '')}".strip()
    bm25 = bm25_sentence_scores(sentences, query_text)

    clf_scores = _minmax(classifier_relevance_scores(relevance_clf, sentences, proto_vec))

    combined = (cfg.evidence_bm25_weight * bm25
                + cfg.evidence_cosine_weight * cosine
                + cfg.evidence_classifier_weight * clf_scores)

    order = np.argsort(-combined)[:min(top_m, len(sentences))]
    return [(sentences[i], float(combined[i])) for i in order]


def predict_case(fact_text, bank_emb, cutoff, margin, relevance_clf=None):
    """Top-K statute ranking (score_case + cutoff/margin decision rule) followed
    by multi-signal evidence retrieval for each predicted (Top-K) section."""
    scores, _legacy_evidence, sentences = score_case(fact_text, bank_emb)
    chosen = _predict_from_scores(scores, cutoff, margin)
    results = []
    for c, s in chosen:
        evidence = retrieve_evidence_sentences(sentences, c, bank_emb, relevance_clf, cfg.top_m_evidence)
        results.append({
            "section": c,
            "score": round(float(s), 4),
            "evidence_sentences": [sent for sent, _ in evidence],
            "evidence_scores": [round(sc, 4) for _, sc in evidence],
        })
    return results


# --------------------------------------------------------------------------
# 9. LLM-Based Reasoning Generation: (IPC Section, Selected Evidence
#    Sentences, CoT prompting) -> Qwen -> Explanation.
# --------------------------------------------------------------------------
_qwen = {}


def _load_qwen():
    if "model" not in _qwen:
        from transformers import AutoModelForCausalLM
        _qwen["tok"] = AutoTokenizer.from_pretrained(cfg.qwen_model_name)
        _qwen["model"] = AutoModelForCausalLM.from_pretrained(
            cfg.qwen_model_name, torch_dtype=torch.float16 if USE_AMP else torch.float32).to(DEVICE)
    return _qwen["tok"], _qwen["model"]


def build_cot_prompt(section_code, evidence_sentences, score):
    """Chain-of-Thought prompt: statute + ranked evidence -> (1) identify
    satisfied elements -> (2) connect each evidence sentence -> (3) final
    explanation. Matches Figure 1's "Input to LLM (IPC Section, Selected
    Evidence Sentences, CoT prompting)" box."""
    title = prototype_titles.get(section_code, "")
    statute_text = prototype_texts.get(section_code, "")
    evidence_block = "\n".join(f"  {i + 1}. {sent} (relevance={sc:.3f})"
                                for i, (sent, sc) in enumerate(evidence_sentences)) or "  (no evidence retrieved)"
    return (
        "You are a legal reasoning assistant analysing an Indian Penal Code (IPC) "
        "statute prediction. Think step by step before answering.\n\n"
        f"Candidate section: {section_code}{f' ({title})' if title else ''}\n"
        f"Statute text: {statute_text}\n"
        f"Prototype similarity score: {score:.3f}\n\n"
        f"Evidence sentences retrieved from the case facts (ranked by combined "
        f"BM25 + cosine similarity + classifier relevance):\n{evidence_block}\n\n"
        "Step 1 - List which elements of the statute the evidence appears to satisfy.\n"
        "Step 2 - For each evidence sentence, state briefly how it supports (or "
        "weakens) applicability of this section.\n"
        "Step 3 - Give a final, concise legal explanation (2-4 sentences) linking "
        "the evidence to the statute.\n\n"
        "Respond with your Step 1 and Step 2 reasoning first, then a line "
        "'Final Explanation:' followed by the Step 3 explanation."
    )


def generate_explanation(section_code, evidence_sentences, score):
    """evidence_sentences: list of (sentence_text, combined_relevance_score) tuples,
    already ranked by retrieve_evidence_sentences(). Uses Qwen with CoT prompting
    when cfg.use_qwen_explainer=True; otherwise a deterministic template reproduces
    the same three-step structure without a decoder LLM."""
    if cfg.use_qwen_explainer:
        tok, model = _load_qwen()
        prompt = build_cot_prompt(section_code, evidence_sentences, score)
        messages = [
            {"role": "system", "content": "You are a legal assistant explaining why an IPC section applies to a case."},
            {"role": "user", "content": prompt},
        ]
        chat_prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(chat_prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=cfg.qwen_max_new_tokens, do_sample=False)
        full_output = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        if "Final Explanation:" in full_output:
            return full_output.split("Final Explanation:", 1)[1].strip()
        return full_output

    # Deterministic fallback: same CoT structure, no decoder LLM required.
    title = prototype_titles.get(section_code, "")
    title_part = f" ({title})" if title else ""
    statute_text = prototype_texts.get(section_code, "")
    if not evidence_sentences:
        return (f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}), "
                f"but no evidence sentences were retrieved for this case.")
    top_sent, top_score = evidence_sentences[0]
    others = "; ".join(s[:100] for s, _ in evidence_sentences[1:])
    return (
        f"{section_code}{title_part} is predicted (prototype similarity {score:.3f}). "
        f"Step 1: the case facts describe conduct matching the elements of \"{statute_text[:160]}...\". "
        f"Step 2: the strongest supporting sentence (relevance {top_score:.3f}) is \"{top_sent[:160]}\""
        + (f", further supported by: {others[:200]}." if others else ".")
        + " Step 3 (final explanation): the retrieved evidence, ranked by BM25 + embedding "
          "similarity + classifier relevance, is consistent with this statute applying to the case."
    )


# --------------------------------------------------------------------------
# 10. Evaluation 
# --------------------------------------------------------------------------
def evaluate_run(run_name, cutoff, margin, bank_emb, relevance_clf=None, save_predictions=True):
    predictions = {}
    for d in tqdm(test_split, desc=f"[{run_name}] predicting"):
        preds = predict_case(d["fact"], bank_emb, cutoff, margin, relevance_clf)
        for p in preds:
            evidence_pairs = list(zip(p["evidence_sentences"], p["evidence_scores"]))
            p["explanation"] = generate_explanation(p["section"], evidence_pairs, p["score"])
        predictions[d["doc_id"]] = {"doc_id": d["doc_id"], "statute": preds}

    if save_predictions:
        path = os.path.join(cfg.out_dir, f"predictions_{run_name}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for r in predictions.values():
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print("Saved:", path)

    gold = [d["gold_sections"] for d in test_split]
    pred = [[p["section"] for p in predictions[d["doc_id"]]["statute"]] for d in test_split]

    # Diagnostic: how much would-be noise is out-of-active-set (should be ~0 with the fix,
    # since predict_case now only ever returns codes from active_codes).
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        mlb = MultiLabelBinarizer(classes=active_codes)
        yt, yp = mlb.fit_transform(gold), mlb.transform(pred)
        dropped = [str(w.message) for w in caught if "unknown class" in str(w.message).lower()]
    if dropped:
        print("NOTE: some gold/pred labels fell outside active_codes and were excluded "
              "from macro-F1 (same fixed label set used in calibration):", dropped)

    metrics = {
        "macro_f1": f1_score(yt, yp, average="macro", zero_division=0),
        "micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
        "weighted_f1": f1_score(yt, yp, average="weighted", zero_division=0),
        "macro_precision": precision_score(yt, yp, average="macro", zero_division=0),
        "macro_recall": recall_score(yt, yp, average="macro", zero_division=0),
        "micro_precision": precision_score(yt, yp, average="micro", zero_division=0),
        "micro_recall": recall_score(yt, yp, average="micro", zero_division=0),
        "exact_match_accuracy": accuracy_score(yt, yp),
        "hamming_loss": hamming_loss(yt, yp),
    }
    print(f"\n--- {run_name}: TEST metrics ---")
    for k, v in metrics.items():
        print(f"{k:>22s}: {v:.4f}")
    report = pd.DataFrame(classification_report(yt, yp, target_names=active_codes, zero_division=0, output_dict=True)).T
    report.to_csv(os.path.join(cfg.out_dir, f"per_class_{run_name}.csv"))
    with open(os.path.join(cfg.out_dir, f"metrics_{run_name}.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics, predictions


# --------------------------------------------------------------------------
# 11. Run 1 - prototype-contrastive fine-tuning 
# --------------------------------------------------------------------------
class SentencePrototypePairs(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        return self.pairs[i]


def collate_pairs(batch):
    return [s for s, _ in batch], [c for _, c in batch]


def make_pair_loader(train_pairs):
    if cfg.use_class_balanced_sampling:
        code_counts = Counter(c for _, c in train_pairs)
        weights = [1.0 / code_counts[c] for _, c in train_pairs]
        sampler = WeightedRandomSampler(weights, num_samples=len(train_pairs), replacement=True)
        return DataLoader(SentencePrototypePairs(train_pairs), batch_size=cfg.batch_size,
                           sampler=sampler, collate_fn=collate_pairs)
    return DataLoader(SentencePrototypePairs(train_pairs), batch_size=cfg.batch_size,
                       shuffle=True, collate_fn=collate_pairs)


def lr_lambda_fn(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def train_prototype_contrastive():
    base_bank = build_prototype_bank(encoder)  # full catalog, for hard-negative mining only
    hard_negative_map = mine_hard_negative_prototypes(
        base_bank, cfg.num_hard_negatives, cfg.exclude_same_base_section)
    demo = next((c for c in ("IPC 302", "IPC 498A") if c in hard_negative_map), prototype_codes[0])
    print(f"Hard-negative prototypes for {demo}: {hard_negative_map[demo]}")

    train_pairs = []
    for rec in train_split:
        train_pairs.extend(positive_pairs_from_explanation(
            rec["fact"], rec.get("explanation", {}) or {}, rec["gold_sections"]))
    print(f"{len(train_pairs)} (sentence, positive prototype) training pairs")

    pair_loader = make_pair_loader(train_pairs)
    steps_per_epoch = len(pair_loader)
    total_steps = steps_per_epoch * cfg.num_epochs
    warmup_steps = int(total_steps * cfg.warmup_ratio)

    optimizer = torch.optim.AdamW([p for p in encoder.parameters() if p.requires_grad],
                                   lr=cfg.encoder_lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda s: lr_lambda_fn(s, total_steps, warmup_steps))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    checkpoints = {}
    global_step = 0
    for epoch in range(1, cfg.num_epochs + 1):
        encoder.train()
        running, steps = 0.0, 0
        optimizer.zero_grad()
        for step, (sentences, pos_codes) in enumerate(pair_loader, start=1):
            positives = list(dict.fromkeys(pos_codes))
            negatives = []
            for c in positives:
                for n in hard_negative_map[c]:
                    if n not in positives and n not in negatives:
                        negatives.append(n)
            # + fresh random global negatives each step, for extra diversity
            random_pool = [c for c in prototype_codes if c not in positives and c not in negatives]
            negatives.extend(random.sample(random_pool, min(cfg.num_random_negatives, len(random_pool))))

            batch_prototypes = positives + negatives
            targets = torch.tensor([batch_prototypes.index(c) for c in pos_codes], device=DEVICE)

            with torch.amp.autocast("cuda", enabled=USE_AMP):
                sent_emb = encoder(sentences, cfg.max_length)
                proto_emb = encoder([prototype_texts[c] for c in batch_prototypes], cfg.max_length)
                logits = sent_emb @ proto_emb.t() / cfg.temperature
                loss = F.cross_entropy(logits.float(), targets)

            scaler.scale(loss / cfg.grad_accum_steps).backward()
            if step % cfg.grad_accum_steps == 0 or step == len(pair_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_([p for p in encoder.parameters() if p.requires_grad], cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()
            running += loss.item()
            steps += 1
            global_step += 1

        avg_loss = running / max(1, steps)
        print(f"[prototype-contrastive] epoch {epoch}/{cfg.num_epochs}  InfoNCE loss = {avg_loss:.4f}")
        checkpoints[epoch] = copy.deepcopy(encoder.state_dict())

  
    print("\n[checkpoint search] evaluating validation macro-F1 for every epoch ...")
    best_epoch, best_val_f1, best_cutoff, best_margin = None, -1.0, None, None
    for epoch, state in checkpoints.items():
        encoder.load_state_dict(state)
        if cfg.use_whitening:
            fit_whitening_from_current_encoder()
        bank_emb = build_bank()
        cutoff, margin, val_f1 = calibrate_decision_rule(bank_emb)
        print(f"  epoch {epoch:2d}: val macro-F1 = {val_f1:.4f}  (cutoff={cutoff}, margin={margin})")
        if val_f1 > best_val_f1:
            best_epoch, best_val_f1, best_cutoff, best_margin = epoch, val_f1, cutoff, margin

    print(f"\n[checkpoint search] BEST epoch = {best_epoch}  (val macro-F1 = {best_val_f1:.4f})")
    encoder.load_state_dict(checkpoints[best_epoch])
    if cfg.use_whitening:
        fit_whitening_from_current_encoder()
    torch.save(checkpoints[best_epoch], os.path.join(cfg.out_dir, "prototype_contrastive_encoder_best.pt"))
    return best_cutoff, best_margin


if __name__ == "__main__":
    best_cutoff, best_margin = train_prototype_contrastive()
    bank_emb = build_bank()

    print("\n[relevance classifier] training BM25+cosine+classifier evidence scorer "
          "on the final fine-tuned encoder ...")
    relevance_clf = train_relevance_classifier(bank_emb)

    # prototype pipeline with the CONTRASTIVELY FINE-TUNED InLegalBERT
    # (best epoch selected on validation, over the fixed active-label set),
    # whitened embeddings, a cutoff/margin pair chosen to maximise VAL
    # macro-F1, multi-signal (BM25 + cosine + classifier) evidence retrieval
    # per predicted statute, and CoT-prompted explanation generation.
    run1_metrics, run1_predictions = evaluate_run(
        "run1_prototype_contrastive", best_cutoff, best_margin, bank_emb, relevance_clf)

Device: cuda | AMP: True
ProtoConfig(train_path='task1.jsonl', prototype_source_path='ipc_sections_clean.json', encoder_name='law-ai/InLegalBERT', freeze_layers=0, max_length=384, max_sentences=60, temperature=0.05, num_hard_negatives=8, num_random_negatives=8, exclude_same_base_section=True, encoder_lr=2e-05, weight_decay=0.01, batch_size=8, grad_accum_steps=2, num_epochs=15, warmup_ratio=0.1, grad_clip=1.0, use_class_balanced_sampling=True, train_fraction=0.7, val_fraction=0.1, test_fraction=0.2, use_whitening=True, whitening_dim=256, evidence_per_prototype=2, cutoff_grid=(np.float64(-0.2), np.float64(-0.18), np.float64(-0.16), np.float64(-0.14), np.float64(-0.12), np.float64(-0.1), np.float64(-0.08), np.float64(-0.06), np.float64(-0.04), np.float64(-0.02), np.float64(-0.0), np.float64(0.02), np.float64(0.04), np.float64(0.06), np.float64(0.08), np.float64(0.1), np.float64(0.12), np.float64(0.14), np.float64(0.16), np.float64(0.18), np.float64(0.2), np.float64(0.22), np.float64(0.24)

PySBD:   0%|          | 0/525 [00:00<?, ?it/s]

{'IPC 302': 181, 'IPC 498A': 84, 'IPC 376': 83, 'IPC 420': 80, 'IPC 147': 77, 'IPC 506': 65, 'IPC 201': 51}
Train 371 | Val 51 | Test 103


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hard-negative prototypes for IPC 302: ['IPC 311', 'IPC 395', 'IPC 393', 'IPC 449', 'IPC 303', 'IPC 306', 'IPC 450', 'IPC 325']
2569 (sentence, positive prototype) training pairs


/tmp/ipykernel_26740/182938368.py:956: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[prototype-contrastive] epoch 1/15  InfoNCE loss = 2.9774
[prototype-contrastive] epoch 2/15  InfoNCE loss = 0.9339
[prototype-contrastive] epoch 3/15  InfoNCE loss = 0.5443
[prototype-contrastive] epoch 4/15  InfoNCE loss = 0.4042
[prototype-contrastive] epoch 5/15  InfoNCE loss = 0.2678
[prototype-contrastive] epoch 6/15  InfoNCE loss = 0.2084
[prototype-contrastive] epoch 7/15  InfoNCE loss = 0.1865
[prototype-contrastive] epoch 8/15  InfoNCE loss = 0.1500
[prototype-contrastive] epoch 9/15  InfoNCE loss = 0.1329
[prototype-contrastive] epoch 10/15  InfoNCE loss = 0.1269
[prototype-contrastive] epoch 11/15  InfoNCE loss = 0.1138
[prototype-contrastive] epoch 12/15  InfoNCE loss = 0.0932
[prototype-contrastive] epoch 13/15  InfoNCE loss = 0.0813
[prototype-contrastive] epoch 14/15  InfoNCE loss = 0.0711
[prototype-contrastive] epoch 15/15  InfoNCE loss = 0.0632

[checkpoint search] evaluating validation macro-F1 for every epoch ...
  epoch  1: val macro-F1 = 0.4395  (cutoff=-0.2, mar

[run1_prototype_contrastive] predicting:   0%|          | 0/103 [00:00<?, ?it/s]

Saved: ./prototype_contrastive_outputs/predictions_run1_prototype_contrastive.jsonl

--- run1_prototype_contrastive: TEST metrics ---
              macro_f1: 0.4983
              micro_f1: 0.5323
           weighted_f1: 0.4959
       macro_precision: 0.5305
          macro_recall: 0.5566
       micro_precision: 0.5323
          micro_recall: 0.5323
  exact_match_accuracy: 0.4466
          hamming_loss: 0.1609
